# Transcripción de entrevistas en español con Whisper (PyTorch)

Cuaderno para transcribir entrevistas y grupos focales en español con **Whisper**, el modelo abierto de reconocimiento de voz de OpenAI, que corre sobre **PyTorch** dentro de Colab.

**Qué produce, por cada audio**
- `.docx`: transcripción con marcas de tiempo y, si activas la separación de hablantes, quién habla.
- `.txt`: texto plano con marcas de tiempo.
- `.srt` y `.vtt`: subtítulos sincronizados con el audio, útiles para revisar la transcripción en un reproductor o para importarla junto al audio en software de análisis cualitativo que admita estos formatos.

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución* → **GPU T4**. Sin GPU funciona, pero puede tardar más que la duración del audio.
2. Ejecuta las celdas en orden con ▶.

**Sobre los datos.** Colab se ejecuta en servidores de Google: el audio no se envía a OpenAI, pero sí se procesa en la nube de Google. Revisa que eso sea compatible con el consentimiento informado de tus participantes y con lo aprobado por tu comité de ética. Para máxima confidencialidad, el mismo código puede ejecutarse en una computadora propia con GPU.

**Whisper se equivoca.** Puede omitir frases, confundir nombres propios o, en silencios largos, inventar texto. Toda transcripción debe revisarse escuchando el audio antes de analizarla.

## 1. Instalar y comprobar

In [ ]:
!pip -q install -U openai-whisper python-docx

import torch, whisper
print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Configuración

- **MODELO**: `large-v3-turbo` es rápido y preciso (recomendado). `large-v3` es algo más preciso pero bastante más lento. `medium` sirve si no hay GPU.
- **CONTEXTO**: una o dos frases con nombres propios, siglas y vocabulario técnico de la entrevista. Ayuda a Whisper a escribirlos bien.
- **SEUDONIMOS**: nombres reales que aparecen en el audio y el código con el que se reemplazan en todos los archivos de salida.

In [ ]:
MODELO = "large-v3-turbo"

CONTEXTO = (
    "Entrevista semiestructurada a estudiantes universitarios sobre el uso de la "
    "inteligencia artificial en la investigación: autoeficacia, compromiso académico, "
    "ChatGPT, Claude, marco teórico, Atlas.ti."
)

# Nombre real (como se escucharía en el audio) -> seudónimo o código
SEUDONIMOS = {
    # "María": "E1",
    # "Profesor Gómez": "P1",
}

# Separación de hablantes (opcional, ver sección 6)
SEPARAR_HABLANTES = False
NUM_HABLANTES = 2   # entrevistador + entrevistado; ajústalo para grupos focales

CARPETA_SALIDA = "transcripciones"

## 3. Subir los audios

Acepta `.mp3`, `.m4a`, `.wav`, `.mp4`, `.ogg` y otros formatos comunes. Puedes subir varios a la vez.

Para audios largos o muchos archivos, es más cómodo montar Google Drive: descomenta las líneas indicadas y escribe la ruta de tu carpeta.

In [ ]:
import os, glob
from google.colab import files

os.makedirs("audios", exist_ok=True)

# Opción A: subir desde tu computadora
subidos = files.upload()
for nombre, contenido in subidos.items():
    with open(os.path.join("audios", nombre), "wb") as f:
        f.write(contenido)

# Opción B: usar una carpeta de Google Drive (descomenta)
# from google.colab import drive
# drive.mount("/content/drive")
# CARPETA_AUDIOS = "/content/drive/MyDrive/entrevistas"
# audios = sorted(glob.glob(os.path.join(CARPETA_AUDIOS, "*")))

audios = sorted(glob.glob("audios/*"))
print(f"{len(audios)} audio(s):", *[os.path.basename(a) for a in audios], sep="\n  ")

## 4. Funciones de apoyo

Formateo de tiempos, seudonimización y exportación. No hace falta modificar esta celda.

In [ ]:
import re, datetime
from docx import Document
from docx.shared import Pt, RGBColor

def hms(seg, ms=False, sep=","):
    total_ms = int(round(seg * 1000))
    h, resto = divmod(total_ms, 3_600_000)
    m, resto = divmod(resto, 60_000)
    s, mil = divmod(resto, 1000)
    base = f"{h:02d}:{m:02d}:{s:02d}"
    return f"{base}{sep}{mil:03d}" if ms else base

def seudonimizar(texto, tabla):
    for real, codigo in sorted(tabla.items(), key=lambda x: -len(x[0])):
        texto = re.sub(rf"\b{re.escape(real)}\b", codigo, texto, flags=re.IGNORECASE)
    return texto

def preparar_segmentos(resultado, tabla):
    segs = []
    for s in resultado["segments"]:
        texto = seudonimizar(s["text"].strip(), tabla)
        if texto:
            segs.append({"inicio": s["start"], "fin": s["end"], "texto": texto,
                         "hablante": s.get("hablante")})
    return segs

def exportar_txt(segs, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        for s in segs:
            quien = f"{s['hablante']}: " if s["hablante"] else ""
            f.write(f"[{hms(s['inicio'])}] {quien}{s['texto']}\n")

def exportar_srt(segs, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        for i, s in enumerate(segs, 1):
            quien = f"{s['hablante']}: " if s["hablante"] else ""
            f.write(f"{i}\n{hms(s['inicio'], True)} --> {hms(s['fin'], True)}\n{quien}{s['texto']}\n\n")

def exportar_vtt(segs, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        f.write("WEBVTT\n\n")
        for s in segs:
            quien = f"<v {s['hablante']}>" if s["hablante"] else ""
            f.write(f"{hms(s['inicio'], True, '.')} --> {hms(s['fin'], True, '.')}\n{quien}{s['texto']}\n\n")

def exportar_docx(segs, ruta, nombre_audio, duracion):
    doc = Document()
    estilo = doc.styles["Normal"]
    estilo.font.name = "Calibri"
    estilo.font.size = Pt(11)
    doc.add_heading(f"Transcripción: {os.path.splitext(nombre_audio)[0]}", level=1)
    meta = doc.add_table(rows=0, cols=2)
    meta.style = "Light Grid Accent 1"
    for k, v in [("Archivo de audio", nombre_audio),
                 ("Duración", hms(duracion)),
                 ("Modelo", f"Whisper {MODELO} (PyTorch {torch.__version__})"),
                 ("Fecha de transcripción", datetime.date.today().isoformat()),
                 ("Estado", "Transcripción automática pendiente de revisión")]:
        fila = meta.add_row().cells
        fila[0].text, fila[1].text = k, v
    doc.add_paragraph()
    hablante_previo = None
    for s in segs:
        p = doc.add_paragraph()
        t = p.add_run(f"[{hms(s['inicio'])}] ")
        t.font.color.rgb = RGBColor(0x80, 0x80, 0x80)
        t.font.size = Pt(9)
        if s["hablante"] and s["hablante"] != hablante_previo:
            p.add_run(f"{s['hablante']}: ").bold = True
        hablante_previo = s["hablante"]
        p.add_run(s["texto"])
    doc.save(ruta)

print("Funciones listas.")

## 5. Transcribir

La primera vez se descarga el modelo (1–2 minutos). Con GPU T4 y `large-v3-turbo`, una hora de audio suele tardar pocos minutos.

In [ ]:
import time

modelo = whisper.load_model(MODELO, device=device)
resultados = {}

for ruta in audios:
    nombre = os.path.basename(ruta)
    print(f"Transcribiendo {nombre}...")
    t0 = time.time()
    res = modelo.transcribe(
        ruta,
        language="es",
        initial_prompt=CONTEXTO,
        fp16=(device == "cuda"),
        condition_on_previous_text=False,  # reduce repeticiones e invenciones en silencios largos
    )
    resultados[nombre] = res
    duracion = res["segments"][-1]["end"] if res["segments"] else 0
    print(f"  listo: {len(res['segments'])} segmentos, {hms(duracion)} de audio, "
          f"{time.time() - t0:.0f} s de proceso")

## 6. (Opcional) Separar hablantes

Usa **pyannote**, otro modelo sobre PyTorch, para identificar quién habla en cada momento. Requiere:
1. Una cuenta gratuita en [huggingface.co](https://huggingface.co).
2. Aceptar las condiciones de uso de `pyannote/speaker-diarization-3.1` y `pyannote/segmentation-3.0` en sus páginas de Hugging Face.
3. Crear un token de lectura en *Settings → Access Tokens* y pegarlo cuando la celda lo pida.

Pon `SEPARAR_HABLANTES = True` en la sección 2 y ejecuta esta celda. Los hablantes salen como `HABLANTE_1`, `HABLANTE_2`, etc.; después puedes reemplazarlos por `Entrevistador` y el código del participante en la celda siguiente. La separación automática comete errores, sobre todo con voces que se superponen.

In [ ]:
if SEPARAR_HABLANTES:
    !pip -q install pyannote.audio==3.3.2
    from getpass import getpass
    from pyannote.audio import Pipeline

    token = getpass("Token de Hugging Face: ")
    diarizador = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=token)
    diarizador.to(torch.device(device))

    for ruta in audios:
        nombre = os.path.basename(ruta)
        print(f"Separando hablantes en {nombre}...")
        wav = f"/tmp/{os.path.splitext(nombre)[0]}.wav"
        !ffmpeg -y -loglevel error -i "{ruta}" -ac 1 -ar 16000 "{wav}"
        diar = diarizador(wav, num_speakers=NUM_HABLANTES)
        turnos = [(t.start, t.end, h) for t, _, h in diar.itertracks(yield_label=True)]
        etiquetas = {h: f"HABLANTE_{i + 1}" for i, h in enumerate(sorted({h for *_, h in turnos}))}
        for s in resultados[nombre]["segments"]:
            solape = {}
            for ini, fin, h in turnos:
                d = min(s["end"], fin) - max(s["start"], ini)
                if d > 0:
                    solape[h] = solape.get(h, 0) + d
            s["hablante"] = etiquetas[max(solape, key=solape.get)] if solape else None
    print("Listo.")
else:
    print("Separación de hablantes desactivada (SEPARAR_HABLANTES = False).")

In [ ]:
# Nombres finales de los hablantes (escucha unos segundos para identificar quién es quién)
NOMBRES_HABLANTES = {
    # "HABLANTE_1": "Entrevistador",
    # "HABLANTE_2": "E1",
}
for res in resultados.values():
    for s in res["segments"]:
        if s.get("hablante"):
            s["hablante"] = NOMBRES_HABLANTES.get(s["hablante"], s["hablante"])

## 7. Exportar y descargar

In [ ]:
import shutil

os.makedirs(CARPETA_SALIDA, exist_ok=True)
for nombre, res in resultados.items():
    base = os.path.join(CARPETA_SALIDA, os.path.splitext(nombre)[0])
    segs = preparar_segmentos(res, SEUDONIMOS)
    duracion = res["segments"][-1]["end"] if res["segments"] else 0
    exportar_docx(segs, base + ".docx", nombre, duracion)
    exportar_txt(segs, base + ".txt")
    exportar_srt(segs, base + ".srt")
    exportar_vtt(segs, base + ".vtt")
    print(f"{nombre}: exportado (.docx, .txt, .srt, .vtt)")

shutil.make_archive("transcripciones", "zip", CARPETA_SALIDA)
files.download("transcripciones.zip")

In [ ]:
# Vista previa de la primera transcripción
primera = sorted(glob.glob(os.path.join(CARPETA_SALIDA, "*.txt")))[0]
print(open(primera, encoding="utf-8").read()[:3000])

## Revisión y reporte

**Revisión.** Escucha el audio completo con la transcripción al lado y corrige nombres propios, siglas, palabras inaudibles (márcalas como `[inaudible 00:12:34]`) y cualquier frase que Whisper haya inventado en silencios. Verifica que la seudonimización haya cubierto todas las menciones, incluidas lugares e instituciones que permitan identificar a alguien.

**Al terminar.** Borra los audios y transcripciones del entorno de Colab (se eliminan al cerrar la sesión, pero no de tu Drive si lo montaste).

**Cómo reportarlo en el método** (ejemplo):
> Las entrevistas se transcribieron de forma automática con Whisper (modelo large-v3-turbo; Radford et al., 2023), ejecutado en Google Colab, y cada transcripción fue revisada íntegramente por el investigador contrastándola con el audio. Los nombres de participantes se reemplazaron por códigos durante la transcripción.

Radford, A., Kim, J. W., Xu, T., Brockman, G., McLeavey, C., & Sutskever, I. (2023). Robust speech recognition via large-scale weak supervision. *Proceedings of the 40th International Conference on Machine Learning*, PMLR 202, 28492–28518.